# Chapter 21: Emit

> *"Much of my work has come from being lazy. I didn't like writing raw
> machine code, and so I started work on a programming system that could
> write the instructions for me."*  
> — John Backus, inventor of FORTRAN and BNF, Turing Award Lecture
> (1977)

------------------------------------------------------------------------

### 21.1 The Final Frontier: Target Code Generation

We have arrived at the fourth and final stage of our compiler project.
Having spent a lot of effort engineering the earlier parts of the
compiler, writing the code for actually emitting (outputting) the target
C++ source code may seem like an anticlimax! It is a truism that the
ultimate goal was always code transformation, and not code analysis, and
here we finally realize the fruits of our labours.

Over the preceding three chapters, we engineered the compiler's
front-end and middle-end:

1.  **Chapter 18 (Lexer):** Scanned raw source text into discrete
    `Token` streams.
2.  **Chapter 19 (Parser):** Transformed linear token sequences into
    structured Abstract Syntax Tree (AST) hierarchies using Pratt
    operator-precedence parsing.
3.  **Chapter 20 (Check):** Evaluated lexical scopes via cactus-stack
    symbol tables and statically proved semantic correctness using
    bidirectional type judgements.

<figure id="fig:steps-to-generation">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI0NDBwdCIgaGVpZ2h0PSIzMzNwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCA0NDAuMDAgMzMzLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDMy
OSkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gc291cmNlIC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNz
PSJub2RlIj4KPHRpdGxlPnNvdXJjZTwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIHBvaW50cz0iNDMyLC0zMjUgMCwtMzI1IDAsLTI4MiA0MzIsLTI4MiA0MzIsLTMy
NSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjgiIHk9Ii0zMTIuNSIgZm9udC1mYW1p
bHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0i
MTAuMDAiPkZhYmVyZ2UgU291cmNlIENvZGUgKCZxdW90O3Byb2dyYW0uZmFiJnF1b3Q7KTo8L3Rl
eHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iOCIgeT0iLTI4OS41IiBmb250LWZhbWls
eT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIx
MC4wMCI+bGV0IGRvdWJsZSA9IGZuICh4OiBJbnQpOiBJbnQgPSZndDsgeCAqIDI7PC90ZXh0Pgo8
L2c+CjwhLS0gdG9rZW5zIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRpdGxlPnRv
a2VuczwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0i
NDMyLC0yNTUgMCwtMjU1IDAsLTIzMyA0MzIsLTIzMyA0MzIsLTI1NSIvPgo8dGV4dCB0ZXh0LWFu
Y2hvcj0ic3RhcnQiIHg9IjgiIHk9Ii0yNDEuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBE
ZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPlRva2VuIFN0cmVh
bSAoc3RkOjp2ZWN0b3ImbHQ7VG9rZW4mZ3Q7KTwvdGV4dD4KPC9nPgo8IS0tIHNvdXJjZSYjNDU7
Jmd0O3Rva2VucyAtLT4KPGcgaWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5zb3VyY2Um
IzQ1OyZndDt0b2tlbnM8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBk
PSJNMjE2LC0yODEuNzRDMjE2LC0yNzYuMzkgMjE2LC0yNzAuNjQgMjE2LC0yNjUuMzIiLz4KPHBv
bHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMjE5LjUsLTI2NS4yNSAy
MTYsLTI1NS4yNSAyMTIuNSwtMjY1LjI1IDIxOS41LC0yNjUuMjUiLz4KPHRleHQgdGV4dC1hbmNo
b3I9InN0YXJ0IiB4PSIyMTYiIHk9Ii0yNjYiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVq
YVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj4gJiMxNjA7U3RhZ2Ug
MTogTGV4aWNhbCBBbmFseXNpczwvdGV4dD4KPC9nPgo8IS0tIGFzdCAtLT4KPGcgaWQ9Im5vZGUz
IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5hc3Q8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBz
dHJva2U9ImJsYWNrIiBwb2ludHM9IjQzMiwtMjA2IDAsLTIwNiAwLC0xODQgNDMyLC0xODQgNDMy
LC0yMDYiLz4KPHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSI4IiB5PSItMTkyLjUiIGZvbnQt
ZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNp
emU9IjEwLjAwIj5BYnN0cmFjdCBTeW50YXggVHJlZSAoQVNUTm9kZSBIaWVyYXJjaHkpPC90ZXh0
Pgo8L2c+CjwhLS0gdG9rZW5zJiM0NTsmZ3Q7YXN0IC0tPgo8ZyBpZD0iZWRnZTIiIGNsYXNzPSJl
ZGdlIj4KPHRpdGxlPnRva2VucyYjNDU7Jmd0O2FzdDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUi
IHN0cm9rZT0iYmxhY2siIGQ9Ik0yMTYsLTIzMi41N0MyMTYsLTIyNy43NiAyMTYsLTIyMS44NyAy
MTYsLTIxNi4yNCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRz
PSIyMTkuNSwtMjE2LjA4IDIxNiwtMjA2LjA4IDIxMi41LC0yMTYuMDggMjE5LjUsLTIxNi4wOCIv
Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjIxNiIgeT0iLTIxNyIgZm9udC1mYW1pbHk9
IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAu
MDAiPiAmIzE2MDtTdGFnZSAyOiBTeW50YWN0aWMgQW5hbHlzaXM8L3RleHQ+CjwvZz4KPCEtLSB2
ZXJpZmllZCAtLT4KPGcgaWQ9Im5vZGU0IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT52ZXJpZmllZDwv
dGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iNDMyLC0x
NTcgMCwtMTU3IDAsLTEzNSA0MzIsLTEzNSA0MzIsLTE1NyIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0i
c3RhcnQiIHg9IjgiIHk9Ii0xNDMuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUg
U2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPkRlY29yYXRlZCBhbmQgVmVy
aWZpZWQgVHlwZWQgQVNUPC90ZXh0Pgo8L2c+CjwhLS0gYXN0JiM0NTsmZ3Q7dmVyaWZpZWQgLS0+
CjxnIGlkPSJlZGdlMyIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+YXN0JiM0NTsmZ3Q7dmVyaWZpZWQ8
L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMjE2LC0xODMuNTdD
MjE2LC0xNzguNzYgMjE2LC0xNzIuODcgMjE2LC0xNjcuMjQiLz4KPHBvbHlnb24gZmlsbD0iYmxh
Y2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMjE5LjUsLTE2Ny4wOCAyMTYsLTE1Ny4wOCAyMTIu
NSwtMTY3LjA4IDIxOS41LC0xNjcuMDgiLz4KPHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSIy
MTYiIHk9Ii0xNjgiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywg
bW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj4gJiMxNjA7U3RhZ2UgMzogU2VtYW50aWMgQW5h
bHlzaXM8L3RleHQ+CjwvZz4KPCEtLSBlbWl0IC0tPgo8ZyBpZD0ibm9kZTUiIGNsYXNzPSJub2Rl
Ij4KPHRpdGxlPmVtaXQ8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNr
IiBwb2ludHM9IjQzMiwtOTcgMCwtOTcgMCwtMzkgNDMyLC0zOSA0MzIsLTk3Ii8+Cjx0ZXh0IHRl
eHQtYW5jaG9yPSJzdGFydCIgeD0iOCIgeT0iLTgyIiBmb250LWZhbWlseT0iQ291cmllciBOZXcs
IERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+VGhlIENvZGUg
RW1pdHRlciAoQVNUIFZpc2l0b3IpPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9
IjgiIHk9Ii03MSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBt
b25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPiYjNDU7IExvd2VycyBmdW5jdGlvbmFsIEFTVCBu
b2RlcyB0byB0YXJnZXQgbGFuZ3VhZ2UgY29uc3RydWN0czwvdGV4dD4KPHRleHQgdGV4dC1hbmNo
b3I9InN0YXJ0IiB4PSI4IiB5PSItNjAiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1
IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj4mIzQ1OyBFbWl0cyBpZGlv
bWF0aWMsIHN0cmljdGx5IHR5cGVkIEMrKzIzIHNvdXJjZSBjb2RlPC90ZXh0Pgo8dGV4dCB0ZXh0
LWFuY2hvcj0ic3RhcnQiIHg9IjgiIHk9Ii00OSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBE
ZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPiYjNDU7IFByZXNl
cnZlcyBsZXhpY2FsIHNjb3BlLCBleHByZXNzaW9uIHNlbWFudGljcywgYW5kIGZvcm1hdHRpbmc8
L3RleHQ+CjwvZz4KPCEtLSB2ZXJpZmllZCYjNDU7Jmd0O2VtaXQgLS0+CjxnIGlkPSJlZGdlNCIg
Y2xhc3M9ImVkZ2UiPgo8dGl0bGU+dmVyaWZpZWQmIzQ1OyZndDtlbWl0PC90aXRsZT4KPHBhdGgg
ZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTIxNiwtMTM0Ljc1QzIxNiwtMTI3LjUyIDIx
NiwtMTE3LjM0IDIxNiwtMTA3LjE0Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJs
YWNrIiBwb2ludHM9IjIxOS41LC0xMDcuMDUgMjE2LC05Ny4wNSAyMTIuNSwtMTA3LjA1IDIxOS41
LC0xMDcuMDUiLz4KPHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSIyMTYiIHk9Ii0xMTkiIGZv
bnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250
LXNpemU9IjEwLjAwIj4gJiMxNjA7U3RhZ2UgNDogQ29kZSBHZW5lcmF0aW9uPC90ZXh0Pgo8dGV4
dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjIxNiIgeT0iLTEwOCIgZm9udC1mYW1pbHk9IkNvdXJp
ZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPiAm
IzE2MDsoVGhpcyBDaGFwdGVyKTwvdGV4dD4KPC9nPgo8IS0tIGNvZGUgLS0+CjxnIGlkPSJub2Rl
NiIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+Y29kZTwvdGl0bGU+Cjx0ZXh0IHRleHQtYW5jaG9yPSJt
aWRkbGUiIHg9IjIxNiIgeT0iLTguNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUg
U2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPlRhcmdldCBDKysgQXJ0aWZh
Y3QgKCZxdW90O3Byb2dyYW0uY3BwJnF1b3Q7KTwvdGV4dD4KPC9nPgo8IS0tIGVtaXQmIzQ1OyZn
dDtjb2RlIC0tPgo8ZyBpZD0iZWRnZTUiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmVtaXQmIzQ1OyZn
dDtjb2RlPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTIxNiwt
MzguODZDMjE2LC0zNi43MyAyMTYsLTM0LjYxIDIxNiwtMzIuNTQiLz4KPHBvbHlnb24gZmlsbD0i
YmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMjE5LjUsLTMyLjMxIDIxNiwtMjIuMzEgMjEy
LjUsLTMyLjMxIDIxOS41LC0zMi4zMSIvPgo8L2c+CjwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 21–1: Stages toward C++ Code Emission</figcaption>
</figure>

The final task of a compiler is **Target Code Generation (Emission)**.
The code generator traverses the verified, decorated AST and synthesises
an equivalent program in a target language.

#### 21.1.1 Transpilation and High-Level Target Emission

Compilers target a wide variety of execution substrates:

- **Machine Code / Assembly:** Direct CPU instructions (x86_64, ARM64,
  RISC-V) via back-ends like LLVM.
- **Virtual Machine Bytecode:** Stack-based or register-based bytecode
  (Java Virtual Machine bytecode, WebAssembly, Common Intermediate
  Language).
- **High-Level Target Code (Source-to-Source / Transpilation):**
  Translating one high-level programming language into another (e.g.,
  TypeScript to JavaScript via `tsc`, C++ to JavaScript/Wasm via
  Emscripten, Elm to JavaScript, Nim to C).

Transpiling our functional language **Faberge** into **C++23** provides
an ideal balance of pedagogical clarity and operational power. Modern
C++ possesses first-class lambda expressions, strict static typing, and
runs almost everywhere.

### 21.2 Semantic Mapping: Functional Faberge to C++

To generate valid C++, we must define an exact structural and semantic
mapping for every node in the **Faberge** Abstract Syntax Tree.

<figure>
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIyMCAyMCA4ODAgMzMwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIzMzAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkFyaWFsIE5hcnJvdyIsIEZyZWVTYW5zLCBzYW5zLXNlcmlm
OyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0iYm94U2hhZG93IiB4PSItNSUiIHk9Ii01
JSIgd2lkdGg9IjExMCUiIGhlaWdodD0iMTIwJSI+CiAgICAgIDxmZURyb3BTaGFkb3cgZHg9IjAi
IGR5PSIzIiBzdGREZXZpYXRpb249IjMiIGZsb29kLWNvbG9yPSJ2YXIoLS1zdmctZmcpIiBmbG9v
ZC1vcGFjaXR5PSIwLjUiLz4KICAgIDwvZmlsdGVyPgogIDwvZGVmcz4KICA8IS0tIExlZnQ6IEZh
YmVyZ2UgU291cmNlIENvbnN0cnVjdHMgLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUoNDAs
IDMwKSI+CiAgICA8cmVjdCB4PSIwIiB5PSIwIiB3aWR0aD0iMzgwIiBoZWlnaHQ9IjMwMCIgcng9
IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIg
c3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIGZpbHRlcj0idXJsKCNib3hT
aGFkb3cpIi8+CiAgICA8dGV4dCB4PSIxOTAiIHk9IjIzIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBm
b250LXNpemU9IjE2IiBmb250LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+RmFi
ZXJnZSBGdW5jdGlvbmFsIEFTVCBOb2RlPC90ZXh0PgogICAgPCEtLSBOb2RlIDE6IExldCBCaW5k
aW5nIC0tPgogICAgPHRleHQgeD0iMjAiIHk9IjY1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250
LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCI+TGV0U3RtdEFTVDwvdGV4dD4KICAgIDx0ZXh0
IHg9IjIwIiB5PSI4MiIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCI+bGV0IGxp
bWl0OiBJbnQgPSAxMDA7PC90ZXh0PgogICAgPCEtLSBOb2RlIDI6IExhbWJkYSAvIEZ1bmN0aW9u
IC0tPgogICAgPHRleHQgeD0iMjAiIHk9IjExNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1z
aXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiPkxhbWJkYUV4cHJBU1Q8L3RleHQ+CiAgICA8dGV4
dCB4PSIyMCIgeT0iMTMyIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5mbiAo
eDogSW50LCB5OiBJbnQpOiBJbnQgPSZndDsgeCArIHk8L3RleHQ+CiAgICA8IS0tIE5vZGUgMzog
SWYgRXhwcmVzc2lvbiAtLT4KICAgIDx0ZXh0IHg9IjIwIiB5PSIxNjUiIGZpbGw9InZhcigtLXN2
Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIj5JZkV4cHJBU1QgKEV4cHJl
c3Npb24gRm9ybSk8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iMTgyIiBmaWxsPSJ2YXIoLS1z
dmctZmcpIiBmb250LXNpemU9IjE0Ij5pZiBuICZsdDs9IDEgdGhlbiAxIGVsc2UgbiAqIGZhYyhu
IC0gMSk8L3RleHQ+CiAgICA8IS0tIE5vZGUgNDogQmxvY2sgRXhwcmVzc2lvbiAtLT4KICAgIDx0
ZXh0IHg9IjIwIiB5PSIyMTUiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZv
bnQtd2VpZ2h0PSJib2xkIj5CbG9ja0V4cHJBU1QgKFNjb3BpbmcgJmFtcDsgVmFsdWUpPC90ZXh0
PgogICAgPHRleHQgeD0iMjAiIHk9IjIzMiIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXpl
PSIxNCI+eyBsZXQgdCA9IHggKiAyOyB0ICsgMTAgfTwvdGV4dD4KICAgIDwhLS0gTm9kZSA1OiBQ
cmltaXRpdmUgVHlwZXMgLS0+CiAgICA8dGV4dCB4PSIyMCIgeT0iMjY1IiBmaWxsPSJ2YXIoLS1z
dmctZmcpIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCI+VHlwZU5vZGU8L3RleHQ+
CiAgICA8dGV4dCB4PSIyMCIgeT0iMjgyIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9
IjE0Ij5JbnQgfCBGbG9hdCB8IFN0cmluZyB8IEJvb2wgfCBWb2lkPC90ZXh0PgogIDwvZz4KICA8
IS0tIFRyYW5zZm9ybWF0aW9uIEFycm93cyAtLT4KICA8ZyBzdHJva2U9InZhcigtLXN2Zy1mZyki
IHN0cm9rZS13aWR0aD0iMi41IiBmaWxsPSJub25lIj4KICAgIDxwYXRoIGQ9Ik0gNDMwIDEwMCBM
IDQ5MCAxMDAiLz4KICAgIDxwb2x5Z29uIHBvaW50cz0iNDkwLDEwMCA0ODAsOTUgNDgwLDEwNSIg
ZmlsbD0idmFyKC0tc3ZnLWZnKSIvPgogICAgPHBhdGggZD0iTSA0MzAgMTUwIEwgNDkwIDE1MCIv
PgogICAgPHBvbHlnb24gcG9pbnRzPSI0OTAsMTUwIDQ4MCwxNDUgNDgwLDE1NSIgZmlsbD0idmFy
KC0tc3ZnLWZnKSIvPgogICAgPHBhdGggZD0iTSA0MzAgMjAwIEwgNDkwIDIwMCIvPgogICAgPHBv
bHlnb24gcG9pbnRzPSI0OTAsMjAwIDQ4MCwxOTUgNDgwLDIwNSIgZmlsbD0idmFyKC0tc3ZnLWZn
KSIvPgogICAgPHBhdGggZD0iTSA0MzAgMjUwIEwgNDkwIDI1MCIvPgogICAgPHBvbHlnb24gcG9p
bnRzPSI0OTAsMjUwIDQ4MCwyNDUgNDgwLDI1NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIvPgogICAg
PHBhdGggZD0iTSA0MzAgMzAwIEwgNDkwIDMwMCIvPgogICAgPHBvbHlnb24gcG9pbnRzPSI0OTAs
MzAwIDQ4MCwyOTUgNDgwLDMwNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIvPgogIDwvZz4KICA8IS0t
IFJpZ2h0OiBUYXJnZXQgQysrIENvbnN0cnVjdHMgLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xh
dGUoNTAwLCAzMCkiPgogICAgPHJlY3QgeD0iMCIgeT0iMCIgd2lkdGg9IjM4MCIgaGVpZ2h0PSIz
MDAiIHJ4PSI4IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxi
YWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIiBmaWx0ZXI9InVy
bCgjYm94U2hhZG93KSIvPgogICAgPHJlY3QgeD0iMCIgeT0iMCIgd2lkdGg9IjM4MCIgaGVpZ2h0
PSIzNSIgcng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFs
bGJhY2spKSIvPgogICAgPHRleHQgeD0iMTkwIiB5PSIyMyIgZmlsbD0idmFyKC0tc3ZnLWZnKSIg
Zm9udC1zaXplPSIxNiIgZm9udC13ZWlnaHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPlRh
cmdldCBDKysgT3V0cHV0PC90ZXh0PgogICAgPCEtLSBUUyAxIC0tPgogICAgPHRleHQgeD0iMjAi
IHk9IjY1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0i
Ym9sZCI+Y29uc3QgVmFyaWFibGUgRGVjbGFyYXRpb248L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIg
eT0iODIiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiPmNvbnN0IGludCBsaW1p
dCA9IDEwMDs8L3RleHQ+CiAgICA8IS0tIFRTIDIgLS0+CiAgICA8dGV4dCB4PSIyMCIgeT0iMTE1
IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCI+
VHlwZWQgTW9kZXJuIEMrKyBMYW1iZGE8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iMTMyIiBm
aWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5bJmFtcDtdKGludCB4LCBpbnQgeSkg
LT4gaW50IHsgcmV0dXJuICh4ICsgeSk7IH08L3RleHQ+CiAgICA8IS0tIFRTIDMgLS0+CiAgICA8
dGV4dCB4PSIyMCIgeT0iMTY1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBm
b250LXdlaWdodD0iYm9sZCI+VGVybmFyeSBDb25kaXRpb25hbCBPcGVyYXRvcjwvdGV4dD4KICAg
IDx0ZXh0IHg9IjIwIiB5PSIxODIiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQi
PihuICZsdDs9IDEgPyAxIDogKG4gKiBmYWMobiAtIDEpKSk8L3RleHQ+CiAgICA8IS0tIFRTIDQg
LS0+CiAgICA8dGV4dCB4PSIyMCIgeT0iMjE1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNp
emU9IjE0IiBmb250LXdlaWdodD0iYm9sZCI+SW1tZWRpYXRlbHkgSW52b2tlZCBMYW1iZGEgRXhw
cmVzc2lvbiAoSUlMRSk8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iMjMyIiBmaWxsPSJ2YXIo
LS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5bJmFtcDtdeyBjb25zdCBpbnQgdCA9IHggKiAyOyBy
ZXR1cm4gKHQgKyAxMCk7IH0oKTwvdGV4dD4KICAgIDwhLS0gVFMgNSAtLT4KICAgIDx0ZXh0IHg9
IjIwIiB5PSIyNjUiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2Vp
Z2h0PSJib2xkIj5DKysgUHJpbWl0aXZlIE1hcHBpbmc8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIg
eT0iMjgyIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5pbnQgfCBkb3VibGUg
fCBzdGQ6OnN0cmluZyB8IGJvb2wgfCB2b2lkPC90ZXh0PgogIDwvZz4KPC9zdmc+
" alt="Mapping Faberge to C++" />
<figcaption>Mapping Faberge to C++</figcaption>
</figure>

#### 21.2.1 Type Translation

C++ posesses types which match directly to the source Faberge types, as
shown in Table 21–1.

| Faberge Type | C++ Target Type | Semantic Meaning |
|------------------------|------------------------|------------------------|
| `Int` | `long long` | Numeric value (integral domain, 64-bit) |
| `Float` | `double` | Numeric value (floating-point domain) |
| `String` | `std::string` | UTF-8 character sequence |
| `Bool` | `bool` | Logical truth value (`true` / `false`) |
| `Void` | `void` | Absence of value |
| `(T1, T2 -> Ret)` | Use `auto` to match `[&](T1 p1, T2 p2) -> Ret{...}` | First-class function closure (Modern C++ lambda, `[&]` captures any enclosing scopes) |

Table 21–1: Faberge Types {#tbl:target-types}

#### 21.2.2 Expression vs Statement Discrepancy: The IIFE Pattern

In functional languages like Faberge, **everything is an expression**. A
block `{ let a = 1; let b = 2; a + b }` has a value: the result of its
trailing expression.

In C++, however, curly-brace code blocks `{ ... }` are strictly
**statements** and do not evaluate to values. We cannot write:

``` cpp
// ILLEGAL C++:
const long long x = { const long long a = 1; a + 10; };
```

To bridge this semantic gap, the code emitter wraps Faberge block
expressions inside **Immediately Invoked Function Expressions (IIFEs)**
(also called IILEs, or Immediately Invoked Lambda Expressions in Modern
C++):

``` cpp
// VALID C++ emitted by our compiler:
const long long x = [&]{
    const a = 1;
    return a + 10;
}();
```

This pattern preserves:

1.  **Lexical Scope Isolation:** Variables declared inside the block
    cannot leak to the enclosing outer scope.
2.  **Expression Composability:** The block can appear inline anywhere
    an expression is expected (e.g., inside function arguments, ternary
    branches, or arithmetic operations).

### 21.3 Architecture of the C++ Code Emitter

The code emitter subsystem is designed using Modern C++20 patterns:

- **Visitor Dispatch:** Inherits from `ASTVisitor` to walk the AST in
  preorder and postorder passes.
- **Clean Indentation Buffer:** Manages indentation depth (`indent()`,
  `dedent()`) and formats readable, production-grade C++ with consistent
  spacing.
- **Expression Parenthesisation:** Guarantees arithmetic precedence by
  wrapping complex sub-expressions in grouping parentheses where
  necessary.
- **Entry-Point Synthesis:** Automatically emits runtime driver code to
  execute the `main` function and output results to the console.

<figure>
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIzMCAyMCA4MjAgMjEwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIyMTAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkFyaWFsIE5hcnJvdyIsIEZyZWVTYW5zLCBzYW5zLXNlcmlm
OyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0iZVNoYWRvdyIgeD0iLTUlIiB5PSItNSUi
IHdpZHRoPSIxMTAlIiBoZWlnaHQ9IjEyMCUiPgogICAgICA8ZmVEcm9wU2hhZG93IGR4PSIwIiBk
eT0iMiIgc3RkRGV2aWF0aW9uPSIzIiBmbG9vZC1jb2xvcj0iIzAwMCIgZmxvb2Qtb3BhY2l0eT0i
MC40Ii8+CiAgICA8L2ZpbHRlcj4KICA8L2RlZnM+CiAgPCEtLSBNYWluIFBpcGVsaW5lIEJveCAt
LT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSg0MCwgMzApIj4KICAgIDxyZWN0IHg9IjAiIHk9
IjAiIHdpZHRoPSI4MDAiIGhlaWdodD0iMTkwIiByeD0iOCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQt
MiwgdmFyKC0tc3ZnLXRpbnQtMi1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0
cm9rZS13aWR0aD0iMiIgZmlsdGVyPSJ1cmwoI2VTaGFkb3cpIi8+CiAgICA8dGV4dCB4PSI0MDAi
IHk9IjMwIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE3IiBmb250LXdlaWdodD0i
Ym9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+QysrMjMgQ29kZSBFbWl0dGVyIEJ1ZmZlciBNYW5h
Z2VtZW50PC90ZXh0PgogICAgPCEtLSBTdWItYm94ZXMgLS0+CiAgICA8IS0tIEluZGVudGF0aW9u
IENvbnRyb2xsZXIgLS0+CiAgICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSgzMCwgNjApIj4KICAg
ICAgPHJlY3QgeD0iMCIgeT0iMCIgd2lkdGg9IjIyMCIgaGVpZ2h0PSIxMDAiIHJ4PSI2IiBmaWxs
PSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0i
dmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiLz4KICAgICAgPHRleHQgeD0iMTEwIiB5
PSIyNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNiIgZm9udC13ZWlnaHQ9ImJv
bGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPkluZGVudGF0aW9uIEVuZ2luZTwvdGV4dD4KICAgICAg
PHRleHQgeD0iMjAiIHk9IjUwIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5p
bmRlbnRfbGV2ZWxfIDogaW50PC90ZXh0PgogICAgICA8dGV4dCB4PSIyMCIgeT0iNzAiIGZpbGw9
InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiPmluZGVudCgpICAgPSZndDsgKytsZXZlbDwv
dGV4dD4KICAgICAgPHRleHQgeD0iMjAiIHk9IjkwIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250
LXNpemU9IjE0Ij5kZWRlbnQoKSAgID0mZ3Q7IC0tbGV2ZWw8L3RleHQ+CiAgICA8L2c+CiAgICA8
IS0tIEV4cHJlc3Npb24gU3RyZWFtIC0tPgogICAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUoMjkw
LCA2MCkiPgogICAgICA8cmVjdCB4PSIwIiB5PSIwIiB3aWR0aD0iMjIwIiBoZWlnaHQ9IjEwMCIg
cng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2sp
KSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIvPgogICAgICA8dGV4
dCB4PSIxMTAiIHk9IjI1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE2IiBmb250
LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+RXhwcmVzc2lvbiBGb3JtYXR0ZXI8
L3RleHQ+CiAgICAgIDx0ZXh0IHg9IjIwIiB5PSI1MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9u
dC1zaXplPSIxNCI+c3RkOjpvc3RyaW5nc3RyZWFtIGJ1ZmZlcl88L3RleHQ+CiAgICAgIDx0ZXh0
IHg9IjIwIiB5PSI3MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCI+VGVybmFy
eSBjb25kaXRpb25hbCBsb2dpYzwvdGV4dD4KICAgICAgPHRleHQgeD0iMjAiIHk9IjkwIiBmaWxs
PSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5JSUxFIGJsb2NrIGVtaXNzaW9uPC90ZXh0
PgogICAgPC9nPgogICAgPCEtLSBGaWxlIE91dHB1dCBTdWJzeXN0ZW0gLS0+CiAgICA8ZyB0cmFu
c2Zvcm09InRyYW5zbGF0ZSg1NTAsIDYwKSI+CiAgICAgIDxyZWN0IHg9IjAiIHk9IjAiIHdpZHRo
PSIyMjAiIGhlaWdodD0iMTAwIiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0t
c3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0
aD0iMS41Ii8+CiAgICAgIDx0ZXh0IHg9IjExMCIgeT0iMjUiIGZpbGw9InZhcigtLXN2Zy1mZyki
IGZvbnQtc2l6ZT0iMTYiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5F
bWl0dGVkIFRhcmdldCBBcnRpZmFjdDwvdGV4dD4KICAgICAgPHRleHQgeD0iMjAiIHk9IjUwIiBm
aWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5IZWFkZXIgYW5kIFByZWFtYmxlPC90
ZXh0PgogICAgICA8dGV4dCB4PSIyMCIgeT0iNzAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQt
c2l6ZT0iMTQiPlN5bnRoZXNpc2VkIG1haW4oKSBkcml2ZXI8L3RleHQ+CiAgICAgIDx0ZXh0IHg9
IjIwIiB5PSI5MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCI+RmlsZSBzdHJl
YW06IG91dC5jcHA8L3RleHQ+CiAgICA8L2c+CiAgPC9nPgo8L3N2Zz4=
" alt="C++ Code Output Formatter" />
<figcaption>C++ Code Output Formatter</figcaption>
</figure>

### 21.4 Experimental Implementation in C++: The C++ Code Generator

The listings below complete the Faberge compiler (with reference to all
of the previous C++ code for Faberge):

- Listing 21–1 implements the complete `CppEmitter` class. It walks the
  AST, emits formatted code, and translates types into valid C++ syntax.
- Listing 21–2 unites all four pipeline stages—**Lexer**, **Parser**,
  **TypeChecker**, and **CppEmitter**—into a single compiler driver (see
  Figure 21–2).

<figure id="fig:complete-compiler">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MDguNCIgaGVp
Z2h0PSIxNzUuNiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA2MDguNCAxNzUuNiI+CiAg
PHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwgLnN2Z2Jv
YiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tlLXdpZHRo
OiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0cm9rZS1s
aW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9iIHRleHQg
ewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6IENvdXJp
ZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxM3B4Owp9
Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFuc3Bh
cmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgouc3Zn
Ym9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewogIGZp
bGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxsIHsK
ICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAgbWFy
a2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFtb25k
IHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5lbmRf
bWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9iIC5z
dGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVuX2Np
cmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0
YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Blbl9j
aXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAu
c3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2JpZ19v
cGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJvdyIg
dmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFy
a2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29u
IHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxt
YXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIg
bWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVy
c2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9seWdv
bj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4
IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmll
bnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIy
IiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0i
b3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldp
ZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAg
ICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xl
PgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdCb3g9
IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9
IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9
IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgPC9k
ZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI2MDguNCIg
aGVpZ2h0PSIxNzUuNiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIx
Mi41NSIgeT0iMjcuMzAwMDAxIiA+RmFiZXJnZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSIyNzQuOTUiIHk9IjI3LjMwMDAwMSIgPkNvbXBpbGVyPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM0NS4xNSIgeT0iMjcuMzAwMDAxIiA+RHJpdmVyPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMzLjE1IiB5PSI1OC41IiA+MS48
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzMuMTUiIHk9Ijc0LjEiID4y
LjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMy4xNSIgeT0iODkuNzAw
MDA1IiA+My48L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzMuMTUiIHk9
IjEwNS4zIiA+NC48L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzMuMTUi
IHk9IjEyMC45IiA+NS48L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzMu
MTUiIHk9IjEzNi41IiA+Ni48L3RleHQ+CiAgPGxpbmUgeDE9IjI2NS4yIiB5MT0iNzAuMjAwMDA1
IiB4Mj0iMjgwLjgwMDAyIiB5Mj0iNzAuMjAwMDA1IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8
cG9seWdvbiBwb2ludHM9IjI4MC44MDAwMiw2Ni4zIDI4OC42LDcwLjIwMDAwNSAyODAuODAwMDIs
NzQuMSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSIyNjUuMiIgeTE9Ijg1
LjgiIHgyPSIyODAuODAwMDIiIHkyPSI4NS44IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9s
eWdvbiBwb2ludHM9IjI4MC44MDAwMiw4MS45IDI4OC42LDg1LjggMjgwLjgwMDAyLDg5LjcwMDAw
NSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSIyNjUuMiIgeTE9IjEwMS40
IiB4Mj0iMjgwLjgwMDAyIiB5Mj0iMTAxLjQiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5
Z29uIHBvaW50cz0iMjgwLjgwMDAyLDk3LjUgMjg4LjYsMTAxLjQgMjgwLjgwMDAyLDEwNS4zIiBj
bGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPGxpbmUgeDE9IjI2NS4yIiB5MT0iMTE3IiB4Mj0i
MjgwLjgwMDAyIiB5Mj0iMTE3IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2lu
dHM9IjI4MC44MDAwMiwxMTMuMTAwMDA2IDI4OC42LDExNyAyODAuODAwMDIsMTIwLjkiIGNsYXNz
PSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyOTgu
MzUiIHk9IjEwNS4zIiA+UHJvdmVuPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjM1Mi45NSIgeT0iMTA1LjMiID5UeXBlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjM5MS45NSIgeT0iMTA1LjMiID5TYWZldHk8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMjk4LjM1IiB5PSIxMjAuOSIgPkZvcm1hdHRlZDwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzNzYuMzUiIHk9IjEyMC45IiA+T3V0cHV0PC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQzMC45NSIgeT0iMTIwLjkiID5D
b2RlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjU2LjU1MDAwMyIgeT0i
NTguNSIgPlJlYWQgU291cmNlIEZpbGUgKC5mYWIpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjU2LjU1MDAwMyIgeT0iNzQuMSIgPkxleGVyOjp0b2tlbmlzZSgpPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI5OC4zNSIgeT0iNzQuMSIgPnN0ZDo6
dmVjdG9yJmx0O1Rva2VuJmd0OzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSI1Ni41NTAwMDMiIHk9Ijg5LjcwMDAwNSIgPlBhcnNlcjo6cGFyc2VfcHJvZ3JhbSgpPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI5OC4zNSIgeT0iODkuNzAwMDA1IiA+
c3RkOjp1bmlxdWVfcHRyJmx0O1Byb2dyYW1BU1QmZ3Q7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjU2LjU1MDAwMyIgeT0iMTA1LjMiID5UeXBlQ2hlY2tlcjo6Y2hlY2so
KTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1Ni41NTAwMDMiIHk9IjEy
MC45IiA+Q3BwRW1pdHRlcjo6ZW1pdCgpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjU2LjU1MDAwMyIgeT0iMTM2LjUiID5Xcml0ZSBPdXRwdXQgRmlsZSAoLmNwcCk8L3Rl
eHQ+CiAgPGc+CiAgICA8bGluZSB4MT0iMTEuNzAwMDAxIiB5MT0iNy44IiB4Mj0iNTk2LjciIHky
PSI3LjgiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjExLjcwMDAwMSIgeTE9
IjcuOCIgeDI9IjExLjcwMDAwMSIgeTI9IjE0OC4yIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAg
IDxsaW5lIHgxPSI1OTYuNyIgeTE9IjcuOCIgeDI9IjU5Ni43IiB5Mj0iMTQ4LjIiIGNsYXNzPSJz
b2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjExLjcwMDAwMSIgeTE9IjM5IiB4Mj0iNTk2Ljci
IHkyPSIzOSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iMTEuNzAwMDAxIiB5
MT0iMTQ4LjIiIHgyPSI1OTYuNyIgeTI9IjE0OC4yIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8
L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 21–2: Completed Compiler Driver</figcaption>
</figure>

Listing 21–1: AST Visitor to Emit C++ Code (Emitter.hpp):

In [ ]:
#ifndef EMITTER_HPP
#define EMITTER_HPP

#include "faberge-headers/AST.hpp"
#include "faberge-headers/ASTVisitor.hpp"
#include <sstream>
#include <string>
#include <string_view>
#include <iostream>

namespace Faberge {

class CppEmitter : public ASTVisitor {
public:
    CppEmitter() = default;

    std::string generate(const ProgramAST& program) {
        m_buffer.str("");
        m_buffer.clear();
        m_indent_level = 0;

        // 1. Emit Compiler Header & Preamble
        emit_line("// =============================================================================");
        emit_line("// Generated automatically by the Faberge Compiler (Stage 4 Target Code Emitter)");
        emit_line("// Target: Modern C++23");
        emit_line("// =============================================================================\n");
        emit_line("#include <iostream>");
        emit_line("#include <string>\n");
        emit_line("int main() {");
        indent();

        // 2. Traverse AST and Emit Code
        program.accept(*this);

        // 3. Emit Main Entry-Point Execution Driver
        if (m_has_main) {
            emit_line("std::cout << \"[Faberge Runtime] Program terminated with output: \" << main_() << '\\n';\n");
        }
        emit_line("return 0;");
        dedent();
        emit_line("}");

        return m_buffer.str();
    }

    // -------------------------------------------------------------------------
    // AST Visitor Implementations
    // -------------------------------------------------------------------------

    void visit(const ProgramAST& node) override {
        for (const auto& stmt : node.statements()) {
            stmt->accept(*this);
        }
    }

    void visit(const LetStmtAST& node) override {
        emit_indent();
        m_buffer << "const ";

        if (node.type_annotation().has_value()) {
            m_buffer << map_type(*node.type_annotation()) << " ";
        }
        else {
            m_buffer << "auto ";
        }

        if (node.name() == "main") {
            m_buffer << "main_ = ";
            m_has_main = true;
        }
        else {
            m_buffer << node.name() << " = ";
        }

        std::string prev_self = m_self;
        m_self = node.name();

        node.initialiser().accept(*this);
        m_buffer << ";\n";

        m_self = prev_self;
    }

    void visit(const ExprStmtAST& node) override {
        emit_indent();
        node.expression().accept(*this);
        m_buffer << ";\n";
    }

    void visit(const LiteralExprAST& node) override {
        if (node.token_type() == TokenType::Literal_String) {
            m_buffer << "\"" << escape_string(node.value()) << "\"";
        } else {
            m_buffer << node.value();
        }
    }

    void visit(const VariableExprAST& node) override {
        if (node.name() == m_self) {
            m_buffer << "self_";
        }
        else {
            m_buffer << node.name();
        }
    }

    void visit(const UnaryExprAST& node) override {
        m_buffer << "(";
        if (node.op() == TokenType::Minus) m_buffer << "-";
        else if (node.op() == TokenType::Exclamation) m_buffer << "!";
        
        node.operand().accept(*this);
        m_buffer << ")";
    }

    void visit(const BinaryExprAST& node) override {
        m_buffer << "(";
        node.left().accept(*this);
        m_buffer << " " << map_operator(node.op()) << " ";
        node.right().accept(*this);
        m_buffer << ")";
    }

    void visit(const IfExprAST& node) override {
        // Translate functional if-then-else to C++ ternary expression
        m_buffer << "(";
        node.condition().accept(*this);
        m_buffer << " ? ";
        node.then_branch().accept(*this);
        m_buffer << " : ";
        node.else_branch().accept(*this);
        m_buffer << ")";
    }

    void visit(const CallExprAST& node) override {
        node.callee().accept(*this);
        m_buffer << "(";
        for (size_t i = 0; i < node.arguments().size(); ++i) {
            node.arguments()[i]->accept(*this);
            if (i + 1 < node.arguments().size()) {
                m_buffer << ", ";
            }
        }
        m_buffer << ")";
    }

    void visit(const BlockExprAST& node) override {
        // Emit as an Immediately Invoked Lambda Expression (IILE)
        m_buffer << "([&]{\n";
        indent();

        for (const auto& stmt : node.statements()) {
            stmt->accept(*this);
        }

        if (node.result_expr()) {
            emit_indent();
            m_buffer << "return ";
            node.result_expr()->accept(*this);
            m_buffer << ";\n";
        }

        dedent();
        emit_indent();
        m_buffer << "})()";
    }

    void visit(const LambdaExprAST& node) override {
        m_buffer << "[&](this auto self_";
        for (size_t i = 0; i < node.parameters().size(); ++i) {
            m_buffer << ", " << map_type(node.parameters()[i].type)
                << " " << node.parameters()[i].name;
        }
        m_buffer << ")";

        if (node.return_type().has_value()) {
            m_buffer << " -> " << map_type(*node.return_type()) << " ";
        }

        m_buffer << "{return ";        
        node.body().accept(*this);
        m_buffer << ";}";
    }

private:
    std::ostringstream m_buffer;
    std::string m_self;
    bool m_has_main{false};
    int m_indent_level{0};

    void indent() noexcept { ++m_indent_level; }
    void dedent() noexcept { if (m_indent_level > 0) --m_indent_level; }

    void emit_indent() {
        for (int i = 0; i < m_indent_level; ++i) {
            m_buffer << "    ";
        }
    }

    void emit_line(std::string_view line) {
        emit_indent();
        m_buffer << line << "\n";
    }

    static std::string map_type(const TypeNode& type) {
        switch (type.kind()) {
            case TypeNode::Kind::Primitive:
                switch (type.primitive_kind()) {
                    case PrimitiveTypeKind::Int:    return "long long";
                    case PrimitiveTypeKind::Float:  return "double";
                    case PrimitiveTypeKind::String: return "std::string";
                    case PrimitiveTypeKind::Bool:   return "bool";
                    case PrimitiveTypeKind::Void:   return "void";
                }
                break;
            case TypeNode::Kind::Function:
                // Recursive mapping for higher-order function types
                return "Function";
            case TypeNode::Kind::List:
                return "std::vector<std::any>";
        }
        return "std::any";
    }

    static std::string_view map_operator(TokenType op) noexcept {
        switch (op) {
            case TokenType::Plus:               return "+";
            case TokenType::Minus:              return "-";
            case TokenType::Asterisk:           return "*";
            case TokenType::Slash:              return "/";
            case TokenType::Percent:            return "%";
            case TokenType::EqualEqual:         return "==";
            case TokenType::ExclamationEqual:   return "!=";
            case TokenType::Less:               return "<";
            case TokenType::LessEqual:          return "<=";
            case TokenType::Greater:            return ">";
            case TokenType::GreaterEqual:       return ">=";
            case TokenType::AmpersandAmpersand: return "&&";
            case TokenType::PipePipe:           return "||";
            default:                            return "/* ? */";
        }
    }

    static std::string escape_string(std::string_view raw) {
        std::string out;
        for (char c : raw) {
            if (c == '"') out += "\\\"";
            else if (c == '\\') out += "\\\\";
            else if (c == '\n') out += "\\n";
            else if (c == '\t') out += "\\t";
            else if (c == '\r') out += "\\r";
            else out += c;
        }
        return out;
    }
};

} // namespace Faberge

#endif // EMITTER_HPP

Listing 21–2: Complete Integrated Compiler (main.cpp):

In [ ]:
#include "faberge-headers/Lexer.hpp"
#include "faberge-headers/Parser.hpp"
#include "faberge-headers/TypeChecker.hpp"
#include "faberge-headers/Emitter.hpp"

#include <iostream>
#include <fstream>
#include <sstream>

namespace Faberge {

class Compiler {
public:
    static bool compile_source(std::string_view source_code, 
                               std::string_view filename, 
                               std::ostream& output_stream) {
        std::cout << "\033[1;34m[Faberge Compiler]\033[0m Compiling '" << filename << "'...\n";

        // STAGE 1: Lexical Analysis
        Lexer lexer(source_code, filename);
        auto tokens = lexer.tokenise();
        if (lexer.has_errors()) {
            std::cerr << "\033[1;31m[Compilation Aborted]\033[0m Lexical errors detected.\n";
            return false;
        }
        std::cout << "  ✓ Stage 1: Lexical Analysis completed (" << tokens.size() << " tokens).\n";

        // STAGE 2: Syntactic Analysis (Parser)
        Parser parser(std::move(tokens), source_code);
        auto ast = parser.parse_program();
        if (parser.has_errors()) {
            std::cerr << "\033[1;31m[Compilation Aborted]\033[0m Syntax errors detected.\n";
            return false;
        }
        std::cout << "  ✓ Stage 2: Syntactic Analysis completed (AST constructed).\n";

        // STAGE 3: Semantic Analysis and Static Type Checking
        TypeChecker checker(source_code);
        if (!checker.check(*ast)) {
            std::cerr << "\033[1;31m[Compilation Aborted]\033[0m Semantic type verification failed.\n";
            return false;
        }
        std::cout << "  ✓ Stage 3: Semantic Analysis completed (Type Safety Proved).\n";

        // STAGE 4: Code Generation (C++ Target Emission)
        CppEmitter emitter;
        std::string cpp_code = emitter.generate(*ast);
        std::cout << "  ✓ Stage 4: Target Code Generation completed.\n";

        output_stream << cpp_code;
        return true;
    }
};

} // namespace Faberge

int main() {
    // A complete, expressive functional Faberge program
    constexpr std::string_view faberge_source = R"(
        // --- Mathematical Helper Functions ---
        let square = fn (x: Int): Int => x * x;

        let add = fn (a: Int, b: Int): Int => a + b;

        // Recursive Factorial Implementation
        let factorial = fn (n: Int): Int => {
            if n <= 1 then 
                1 
            else 
                n * factorial(n - 1)
        };

        // Recursive Fibonacci Implementation
        let fibonacci = fn (n: Int): Int => {
            if n <= 0 then
                0
            else if n == 1 then
                1
            else
                fibonacci(n - 1) + fibonacci(n - 2)
        };

        // Main Program Entry Point
        let main = fn (): Int => {
            let num = 5;
            let fac5 = factorial(num);
            let fib7 = fibonacci(7);
            
            // Block Expression with Lexical Scoping
            let combined = {
                let multiplier = 10;
                let squared = square(fac5);
                (squared / multiplier) + fib7
            };

            combined
        };
    )";

    std::cout << "====================================================\n";
    std::cout << "      The Faberge Compiler (Complete Demo)    \n";
    std::cout << "====================================================\n\n";

    std::ostringstream generated_cpp;
    bool success = Faberge::Compiler::compile_source(faberge_source, "sample.fab", generated_cpp);

    if (success) {
        std::cout << "\n\033[1;32m[SUCCESS] Compilation Succeeded!\033[0m\n";
        std::cout << "====================================================\n";
        std::cout << "EMITTED C++ CODE:\n";
        std::cout << "====================================================\n";
        std::cout << generated_cpp.str() << "\n";
        std::cout << "====================================================\n";
    } else {
        std::cout << "\n\033[1;31m[FAILURE] Compilation Failed.\033[0m\n";
        return 1;
    }

    return 0;
}

In [ ]:
main(); // Invocation of main program for Jupyter notebook running the xeus-cpp kernel

### 21.6 Verification: Executing Emitted C++

When our compiler executes the driver above, the **C++ Emitter**
synthesises the following clean, typed C++23 output:

``` cpp
// =============================================================================
// Generated automatically by the Faberge Compiler (Stage 4 Target Code Emitter)
// Target: Modern C++23
// =============================================================================

#include <iostream>
#include <string>

int main() {
    const auto square = [&](this auto self_, long long x) -> long long {return (x * x);};
    const auto add = [&](this auto self_, long long a, long long b) -> long long {return (a + b);};
    const auto factorial = [&](this auto self_, long long n) -> long long {return ([&]{
        return ((n <= 1) ? 1 : (n * self_((n - 1))));
    })();};
    const auto fibonacci = [&](this auto self_, long long n) -> long long {return ([&]{
        return ((n <= 0) ? 0 : ((n == 1) ? 1 : (self_((n - 1)) + self_((n - 2)))));
    })();};
    const auto main_ = [&](this auto self_) -> long long {return ([&]{
        const auto num = 5;
        const auto fac5 = factorial(num);
        const auto fib7 = fibonacci(7);
        const auto combined = ([&]{
            const auto multiplier = 10;
            const auto squared = square(fac5);
            return ((squared / multiplier) + fib7);
        })();
        return combined;
    })();};
    std::cout << "[Faberge Runtime] Program terminated with output: " << main_() << '\n';

    return 0;
}
```

When run, this program outputs:

```
[Faberge Runtime] Program terminated with output: 1453
```

#### 21.6.1 Mathematical Trace Verification

To verify the runtime output step-by-step:

1.  `factorial(5)` $= 5 \times 4 \times 3 \times 2 \times 1 = 120$.
2.  `fibonacci(7)` $= 13$.
3.  `square(fac5)` $= 120^2 = 14,400$.
4.  `squared / multiplier` $= 14,400 / 10 = 1,440$.
5.  `combined` $= 1,440 + 13 = 1,453$.

The emitted code executes with exact mathematical precision, confirming
the end-to-end correctness of our entire compiler pipeline.

### 21.7 Summary

In this chapter we have achieved the following:

- The Visitor Pattern has been used to create a class which generates
  valid C++23 code from the type-checked Abstract Syntax Tree.
- Equivalent types are used in the C++ code to that in the Faberge
  source, and this guarantees that the target program is syntactically
  valid. -A primitive, but fully functional, compiler driver reads a
  valid Faberge program from a string and outputs a valid C++ program
  which can then be compiled into an executable.

With the completion of **Stage 4: Code Generation**, our four-stage
compiler is fully realized.

### 21.8 Programming Challenges

#### Exercise 21.1: Tail-Call Optimisation (TCO) to Iterative Loops

**Context and Theory:**

Recursive algorithms (such as accumulator-based factorial or list
processing) can exhaust the physical operating system call stack when
recursing over large inputs, causing a runtime segmentation fault:

$$\text{Stack Frame Depth} = O(N)$$

While some C++ compilers perform tail-call elimination under `-O2` or
`-O3`, it is not guaranteed by the C++ standard. The Faberge compiler
can enforce **Guaranteed Tail-Call Elimination** at the code generation
level:

1.  Identify recursive calls located strictly in **Tail Position** (the
    final expression evaluated before function return).
2.  Rewrite the tail-recursive function into an iterative C++
    `while (true)` loop:
3.  Update parameter variables in-place and continue looping, reducing
    call-stack consumption to $O(1)$ memory.

Faberge Tail-Recursive AST:

```
let fac = fn (n, acc) =>
    if n <= 1 then acc
    else fac(n - 1, n * acc)
```

Emitted C++ Loop:

``` cpp
const auto fac = [](int64_t n, int64_t acc) {
    while (true) {
        if (n <= 1) return acc;
        acc = n * acc; n = n - 1;
    }
};
```

**Specification:**

Implement an AST analysis pass in `CppEmitter` that detects
tail-recursive lambda functions. Emit an iterative C++ `while (true)`
loop where recursive calls are lowered into parameter mutations and
`continue` statements.

``` cpp
class TailCallOptimizer {
public:
    static bool is_tail_call(const CallExprAST& call, std::string_view current_fn);
    static void emit_tco_loop(const LambdaExprAST& lambda, std::string_view fn_name, 
                             std::ostringstream& out) {
        out << "[";
        emit_parameters_as_mutable_locals(lambda.parameters(), out);
        out << "]() mutable {\n    while (true) {\n";
        emit_body_as_tco_branch(lambda.body(), fn_name, out);
        out << "    }\n}()";
    }
};
```

**Verification Tasks:**

1.  **Stack Overflow Immunity:** Emit a tail-recursive accumulator sum
    function calculating $\sum_{i=1}^{10^7} i$. Run the compiled binary
    and verify that it completes without stack overflow.
2.  **Non-Tail Preservation:** Verify that non-tail recursive functions
    (e.g. `n * fac(n - 1)`) are left untouched as standard function
    calls.
3.  **Multi-Parameter Mutation Order:** Check a function where arguments
    swap positions: `gcd(b, a % b)`. Verify that the emitter generates
    temporary variables during reassignment to prevent clobbering:
    `auto __t0 = b; auto __t1 = a % b; a = __t0; b = __t1; continue;`.

#### Exercise 21.2: Lowering Heterogeneous Tuples to `std::tuple`

**Context and Theory:**

Faberge heterogeneous product types (tuples) must be mapped to
efficient, cache-friendly native structures. In Modern C++, the
canonical representation is `std::tuple`.

The emitter maps tuple operations as follows:

- **Construction:**
  $(e_1, e_2, \dots, e_k) \implies \text{std::make\_tuple}(e_1, e_2, \dots, e_k)$.
- **Component Extraction:** $t.i \implies \text{std::get}<i>(t)$.
- **Destructuring:** Lowering pattern-matched tuple variables into C++17
  **Structured Bindings**:

$$\text{let } (a, b) = \text{pair}; \implies \text{const auto } [a, b] = \text{pair};$$

This lowering guarantees that tuples are stored contiguously in stack
memory without dynamic heap allocations. This allows for compiling the
Faberge code:

```
let p = (42, "Faberge", true); let id = p.0;
```

To C++ target:

``` cpp
const auto p = std::make_tuple(42, std::string("Faberge"), true);
const auto id = std::get<0>(p);
```

**Specification:**

Extend `CppEmitter` to lower `TupleExprAST` and `TupleAccessExprAST`.
Generate the required `#include <tuple>` header directive and emit clean
`std::make_tuple` and `std::get` function calls.

``` cpp
void CppEmitter::visit(const TupleExprAST& node) {
    m_buffer << "std::make_tuple(";
    for (size_t i = 0; i < node.elements().size(); ++i) {
        node.elements()[i]->accept(*this);
        if (i + 1 < node.elements().size()) m_buffer << ", ";
    }
    m_buffer << ")";
}

void CppEmitter::emit_tuple_access(const ExprAST& target, size_t index) {
    m_buffer << "std::get<" << index << ">(";
    target.accept(*this);
    m_buffer << ")";
}
```

**Verification Tasks:**

1.  **Tuple Instantiation and Access:** Emit
    `let t = (10, 3.14); let x = t.0; let y = t.1;`. Compile the output
    and verify that `x` is `int64_t(10)` and `y` is `double(3.14)`.
2.  **Structured Binding Emission:** Emit a destructuring let-binding:
    `let (name, age) = get_user();`. Verify that the emitted C++ matches
    `const auto [name, age] = get_user();`.
3.  **Zero-Tuple / Unit Handling:** Verify that the empty tuple `()`
    emits `std::monostate{}` or `std::tuple<>{}`.

#### Exercise 21.3: Lowering Homogeneous Lists to `std::vector`

**Context and Theory:**

Faberge homogeneous list collections map naturally to Modern C++
`std::vector<T>`.

The code generator must translate operations cleanly:

- **List Construction:**
  $[e_1, e_2, \dots, e_k] \implies \text{std::vector}\langle T\rangle\{e_1, e_2, \dots, e_k\}$.
- **Empty Lists:** Requires explicit type specialization based on
  semantic analysis: `std::vector<int64_t>{}`.
- **Subscript Access:** Translates $arr[i]$ into safe, bounds-checked
  element extraction (`arr.at(static_cast<size_t>(i))`) or unchecked
  direct indexing (`arr[static_cast<size_t>(i)]`).

Because Faberge integers are signed 64-bit (`int64_t`) and C++ container
indices are unsigned (`size_t`), the emitter must insert static casts to
prevent compiler signed/unsigned comparison warnings. This allows for
compiling the Faberge code:

```
[ 10, 20, 30 ][0]
```

To C++ target:

``` cpp
std::vector<int64_t>{10, 20, 30}.at(static_cast<size_t>(0));
```

**Specification:**

Extend `CppEmitter` to handle `ListLiteralExprAST` and `IndexExprAST`.
Retrieve the verified element type from the semantic analysis table,
emit the explicit `std::vector<T>` constructor, and generate static
casts on index access expressions.

``` cpp
void CppEmitter::visit(const ListLiteralExprAST& node) {
    Type elem_t = m_type_table.get_type(node).list_element_type();
    m_buffer << "std::vector<" << map_cpp_type(elem_t) << ">{";
    for (size_t i = 0; i < node.elements().size(); ++i) {
        node.elements()[i]->accept(*this);
        if (i + 1 < node.elements().size()) m_buffer << ", ";
    }
    m_buffer << "}";
}

void CppEmitter::visit(const IndexExprAST& node) {
    node.target().accept(*this);
    m_buffer << ".at(static_cast<size_t>(";
    node.index().accept(*this);
    m_buffer << "))";
}
```

**Verification Tasks:**

1.  **Vector Literal Compilation:** Emit `let xs = [1, 2, 3, 4];`.
    Verify that the emitted C++ compiles cleanly without type inference
    warnings.
2.  **Bounds-Checked Indexing:** Emit `let v = xs[2];`. Compile and run
    the binary; assert that `v == 3`.
3.  **Empty List Type Specialization:** Emit `let empty: [Float] = [];`.
    Verify that the emitter generates `std::vector<double>{}` rather
    than untyped `{}`.

#### Exercise 21.4: AST Constant Folding and `constexpr` Emission

**Context and Theory:**

If an expression consists entirely of compile-time constants (e.g.
`10 + 20 * 3`), evaluating it at runtime wastes CPU cycles.

The Faberge compiler performs **Constant Folding** as an AST-to-AST
optimization pass before code generation:

1.  Traverse the AST in postorder.
2.  If both children of a `BinaryExprAST` are `LiteralExprAST` nodes,
    evaluate the operation at compile time.
3.  Replace the binary operator node with a single computed
    `LiteralExprAST`.
4.  Variables initialized with compile-time pure expressions are emitted
    using C++ **`constexpr`** rather than `const`, shifting execution to
    C++ compile-time evaluation.

- **Before Folding:** `BinaryExpr(+) [ 10, BinaryExpr(*) [ 20, 3 ] ]`
- **After Folding:** `LiteralExprAST(70)`
- **Emitted C++:** `constexpr int64_t total = 70;`

**Specification:**

Implement a `ConstantFoldingPass` that folds arithmetic and boolean
operations. Extend `CppEmitter` to emit `constexpr` for variables marked
as compile-time constants by the optimizer.

``` cpp
class ConstantFoldingPass : public ASTVisitor {
public:
    std::unique_ptr<ExprAST> fold(std::unique_ptr<ExprAST> expr) {
        if (auto* bin = dynamic_cast<BinaryExprAST*>(expr.get())) {
            auto left_folded = fold(bin->extract_left());
            auto right_folded = fold(bin->extract_right());
            if (is_literal(left_folded) && is_literal(right_folded)) {
                return evaluate_constant(bin->op(), *left_folded, *right_folded);
            }
            return std::make_unique<BinaryExprAST>(bin->location(), bin->op(), 
                                                  std::move(left_folded), std::move(right_folded));
        }
        return expr;
    }
};
```

**Verification Tasks:**

1.  **Arithmetic Folding:** Run the constant folder on
    `100 * 2 + 50 / 2`. Verify that the generated C++ emits
    `constexpr int64_t x = 225;`.
2.  **Boolean Logic Folding:** Fold `true && false || true`. Verify that
    the output emits `constexpr bool flag = true;`.
3.  **Partial Folding Safety:** Fold `x + (10 * 0)`. Verify that
    non-constant subtrees are preserved and that the AST simplifies to
    `(x + 0)`.

#### Exercise 21.5: Lowering Pattern Matching to C++ `if-else` Cascades

**Context and Theory:**

Faberge's `match expr with { ... }` construct allows structural matching
against literals, tuples, and wildcards. Because standard C++ does not
possess native pattern matching, the code generator lowers `match` into
an IIFE containing a sequence of **`if-else` branches**.

For each arm:

- **Literal Patterns:** Lowered into equality checks:
  `if (target == literal) { return body; }`.
- **Tuple Destructuring:** Lowered into temporary structured bindings
  with guard checks:
  `if (auto [a, b] = target; a == 0) { return body; }`.
- **Wildcard Pattern (`_`):** Lowered into the unconditional terminal
  `else` block.

Faberge:

```
 match x with {
     0 => "zero",
     _ => "other"
 }
```

Lowered C++:

``` cpp
 [&]() -> std::string {
     const auto __target = x;
     if (__target == 0) { return "zero"; }
     else { return "other"; }
 }()
```

**Specification:**

Implement `emit_match_expression()` in `CppEmitter`. Cache the evaluated
target in a local `const auto __target` variable to prevent repeated
evaluation side-effects, and emit sequential `if-else` branches for each
arm.

``` cpp
void CppEmitter::visit(const MatchExprAST& node) {
    m_buffer << "[&]() -> decltype(auto) {\n";
    m_buffer << "    const auto __target = ";
    node.target().accept(*this);
    m_buffer << ";\n";
    for (size_t i = 0; i < node.arms().size(); ++i) {
        const auto& arm = node.arms()[i];
        if (arm.pattern->is_wildcard()) {
            m_buffer << "    else {\n        return ";
            arm.body->accept(*this);
            m_buffer << ";\n    }\n";
            break;
        }
        m_buffer << (i == 0 ? "    if (" : "    else if (");
        emit_pattern_condition(*arm.pattern, "__target");
        m_buffer << ") {\n        return ";
        arm.body->accept(*this);
        m_buffer << ";\n    }\n";
    }
    m_buffer << "}()";
}
```

**Verification Tasks:**

1.  **Scalar Pattern Lowering:** Lower a match statement over an integer
    with cases `1`, `2`, and `_`. Compile and run the emitted C++
    binary; assert exact matching behaviour.
2.  **Target Side-Effect Invariance:** Match against a function call
    expression: `match get_next_id() with { ... }`. Verify that the
    emitted C++ evaluates `get_next_id()` exactly once into `__target`.
3.  **Tuple Pattern Unpacking:** Lower
    `match pair with { (0, y) => y, _ => -1 }`. Verify that the emitted
    C++ correctly tests `std::get<0>(__target) == 0`.

#### Exercise 21.6: Closure Capture Lowering and Lifetime Management

**Context and Theory:**

When a lambda references variables declared in an enclosing scope, the
code generator must emit explicit C++ lambda capture clauses:

``` cpp
// Target capture strategies:
[&]                // Capture all by reference (stack-only execution)
[=]                // Capture all by value (escaping closure)
[x = std::move(x)] // Move capture
```

Choosing the wrong capture clause leads to catastrophic memory bugs:

- **Dangling References:** If an inner lambda captures local variables
  by reference `[&]` and escapes the stack frame (by being returned or
  stored globally), calling the closure accesses dead stack memory.
- **Unnecessary Copies:** Capturing large vectors or strings by value
  `[=]` in non-escaping closures causes redundant memory allocations.

The emitter queries the **Escape Analysis Table** (computed in Chapter
20, Exercise 12) in order to obtain a result (see Figure 21–3):

1.  **Non-Escaping Lambda:** Emit `[&]` (safe, zero-copy reference).
2.  **Escaping Lambda:** Emit explicit value captures `[x, y]` for all
    captured free variables.

<figure id="fig:escape-analysis">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1OTIuOCIgaGVp
Z2h0PSI2Ni40IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDU5Mi44IDY2LjQiPgogIDxz
dHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5zdmdib2Ig
cmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13aWR0aDog
MS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJva2UtbGlu
ZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0ZXh0IHsK
ICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNweDsKfQoK
LnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNwYXJl
bnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2Z2Jv
YiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7CiAg
ZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1hcmtl
ci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9uZCB7
CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Rp
YW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5kX21h
cmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3Rh
cnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9jaXJj
bGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFy
dDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5fY2ly
Y2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0
YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdfb3Bl
bl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ciIHZp
ZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBw
b2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHlnb24+
CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDggOCIg
cmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50
PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIg
Y2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9Im9w
ZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4K
ICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwvZGVm
cz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNTkyLjgiIGhl
aWdodD0iNjYuNCI+PC9yZWN0PgogIDxsaW5lIHgxPSI0MzYuODAwMDIiIHkxPSI3LjgiIHgyPSI0
NjAuMiIgeTI9IjcuOCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSI0
NjAuMiwzLjkgNDY4LDcuOCA0NjAuMiwxMS43MDAwMDEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdv
bj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0NzcuNzUiIHk9IjExLjcwMDAwMSIg
PkVtaXQ6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjUyNC41NSIgeT0i
MTEuNzAwMDAxIiA+WyZhbXA7XTwvdGV4dD4KICA8bGluZSB4MT0iNDM2LjgwMDAyIiB5MT0iMzki
IHgyPSI0NjAuMiIgeTI9IjM5IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2lu
dHM9IjQ2MC4yLDM1LjEwMDAwMiA0NjgsMzkgNDYwLjIsNDIuOSIgY2xhc3M9ImZpbGxlZCI+PC9w
b2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ3Ny43NSIgeT0iNDIuOSIg
PkVtaXQ6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjUyNC41NSIgeT0i
NDIuOSIgPltmYWN0b3JdPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijku
NzUiIHk9IjExLjcwMDAwMSIgPkxhbWJkYSBBIChQYXNzZWQgdG8gbG9jYWwgYWxnb3JpdGhtKTog
RXNjYXBlcyA9IGZhbHNlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijku
NzUiIHk9IjQyLjkiID5MYW1iZGEgQiAoUmV0dXJuZWQgZnJvbSBmdW5jdGlvbik6ICAgIEVzY2Fw
ZXMgPSB0cnVlPC90ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 21–3: Escape Analysis Result</figcaption>
</figure>

**Specification:**

Update `CppEmitter::visit(const LambdaExprAST&)` to consult the escape
analysis metadata. Select between `[&]` and explicit value capture lists
`[var1, var2]` based on the lambda's escape status.

``` cpp
void CppEmitter::emit_lambda_capture_clause(const LambdaExprAST& node) {
    const auto& meta = m_escape_table.get_metadata(&node);
    if (!meta.escapes_local_frame) {
        m_buffer << "[&]";
        return;
    }
    m_buffer << "[";
    size_t i = 0;
    for (const auto& var_name : meta.captured_variables) {
        m_buffer << var_name;
        if (++i < meta.captured_variables.size()) m_buffer << ", ";
    }
    m_buffer << "]";
}
```

**Verification Tasks:**

1.  **Escaping Closure Execution:** Emit a higher-order adder function:
    `let make_adder = fn (x: Int) => fn (y: Int) => x + y;`. Compile and
    run the emitted C++; verify that the returned closure retains `x`
    safely by value and evaluates correctly.
2.  **Non-Escaping Reference Optimization:** Check a lambda passed to an
    immediate call: `(fn (a: Int) => a + local_var)(5)`. Verify that the
    emitter outputs `[&]` rather than copying `local_var`.
3.  **Empty Capture Clause:** Check a pure lambda referencing only its
    own parameters. Verify that it emits an empty capture clause `[]`.

#### Exercise 21.7: Emitting Header Preambles, Namespaces and Type Aliases

**Context and Theory:**

A generated C++ source file must form a complete, self-contained
**Translation Unit**.

The code generator must emit structured boilerplate:

1.  **Compiler Flags and Standard Attributes:**
    `/* Generated by Faberge Compiler */` header.
2.  **Pragmatic Include Elimination:** The emitter should track language
    features used in the AST and emit *only* the required standard
    headers (e.g. emit `<vector>` only if lists exist, `<tuple>` only if
    tuples exist).
3.  **Namespace Encapsulation:** Wrap generated code in a dedicated C++
    namespace (e.g. `namespace FabergeGenerated`) to prevent collisions
    with C runtime symbols.
4.  **Main Entry-Point Translation:** Generate a native C++
    `int main(int argc, char* argv[])` wrapper that invokes the Faberge
    `main` function and translates its return integer to process exit
    status.

**Specification:**

Create a `CppPreambleBuilder` class. Traverse the AST to gather required
standard library dependencies, emit minimal `#include` directives, wrap
generated logic in `namespace FabergeGenerated`, and synthesize the
top-level `main` driver.

``` cpp
class CppPreambleBuilder {
public:
    void record_feature_usage(const ProgramAST& program);
    void emit_preamble(std::ostringstream& out) const {
        out << "// Generated by the Faberge Compiler (Target: C++23)\n";
        out << "#include <cstdint>\n#include <string>\n#include <iostream>\n";
        if (m_uses_vectors) out << "#include <vector>\n";
        if (m_uses_tuples)  out << "#include <tuple>\n";
        out << "\nnamespace FabergeGenerated {\n\n";
    }
    void emit_epilogue(std::ostringstream& out) const {
        out << "\n} // namespace FabergeGenerated\n\n";
        out << "int main() {\n    return static_cast<int>(FabergeGenerated::main());\n}\n";
    }
private:
    bool m_uses_vectors{false};
    bool m_uses_tuples{false};
};
```

**Verification Tasks:**

1.  **Clean Compilation Invariant:** Emit a complete Faberge script
    through `CppPreambleBuilder`. Compile with
    `clang++ -Wall -Wextra -Werror` and verify that the translation unit
    compiles with zero warnings.
2.  **Minimal Dependency Emission:** Compile a program containing only
    basic integer arithmetic. Verify that `<vector>` and `<tuple>` are
    not emitted in the header section.
3.  **Exit Code Propagation:** Emit a Faberge program whose `main`
    function returns `42`. Compile and run the executable; verify that
    `echo $?` confirms exit status `42`.

#### Exercise 21.8: Source Map Generation (V3 Format)

**Context and Theory:**

When a generated C++ program fails at runtime or triggers a compiler
error, the line numbers reported by `clang++` correspond to the emitted
`.cpp` file, not the original `.fab` source file.

To bridge this gap, modern compilers produce a **Source Map (v3
Specification)** (see Figure 21–4). A source map is a JSON document
mapping coordinates in the generated target file back to coordinates in
the original source:

$$(\text{GenLine}, \text{GenCol}) \longleftrightarrow (\text{SrcLine}, \text{SrcCol}, \text{SrcFile})$$

The mapping coordinates are encoded using **Base64 Variable-Length
Quantities (VLQ)**. When debugging in GDB or LLDB, source directives
(`#line <num> "<file>"`) can be emitted directly into the C++ source,
instructing native debuggers to display the original Faberge source code
during step-debugging.

<figure id="fig:source-map">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2NzguNjAwMDQi
IGhlaWdodD0iNjYuNCIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA2NzguNjAwMDQgNjYu
NCI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IENvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAx
M3B4Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0
cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9
Cgouc3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQg
ewogIGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9m
aWxsIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJv
dyB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9k
aWFtb25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9t
YXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9i
IC5lbmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3Zn
Ym9iIC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsK
fQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNv
cGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFy
a2VyLXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdf
b3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwo
I2JpZ19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJh
cnJvdyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0i
NyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxw
b2x5Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4K
ICAgIDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVm
WT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0
LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwv
cG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtl
ciBpZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwv
Y2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZp
ZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJI
ZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0i
NCIgY3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+
CiAgPC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI2
NzguNjAwMDQiIGhlaWdodD0iNjYuNCI+PC9yZWN0PgogIDxwb2x5Z29uIHBvaW50cz0iMjY1LjIs
My45IDI1Ny40LDcuOCAyNjUuMiwxMS43MDAwMDEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4K
ICA8bGluZSB4MT0iMjY1LjIiIHkxPSI3LjgiIHgyPSIyOTYuNCIgeTI9IjcuOCIgY2xhc3M9InNv
bGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzA2LjE1IiB5PSIx
MS43MDAwMDEiID5Tb3VyY2U8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MzYwLjc1IiB5PSIxMS43MDAwMDEiID5NYXA8L3RleHQ+CiAgPGxpbmUgeDE9IjM5MCIgeTE9Ijcu
OCIgeDI9IjQyMS4yIiB5Mj0iNy44IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBw
b2ludHM9IjQyMS4yLDMuOSA0MjksNy44IDQyMS4yLDExLjcwMDAwMSIgY2xhc3M9ImZpbGxlZCI+
PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUiIHk9IjExLjcw
MDAwMSIgPkdlbmVyYXRlZCBDKysgKExpbmUgMTIsIENvbCA0KTwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSI0MzguNzUiIHk9IjExLjcwMDAwMSIgPkZhYmVyZ2UgU291cmNl
IChMaW5lIDMsIENvbCA4KTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5
Ljc1IiB5PSI0Mi45IiA+R0RCIC8gTExEQiBzdGVwLWRlYnVnZ2VyIGRpc3BsYXlzOiBtYWluLmZh
YjozOjg8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 21–4: Mapping Output Code to Input
Source</figcaption>
</figure>

**Specification:**

Implement a `SourceMapEmitter` class. During C++ code emission, record
mapping tuples whenever an AST node is lowered. Provide an option to
emit C++ `#line` directives directly into the emitted code.

``` cpp
struct MappingCoordinate {
    uint32_t gen_line;
    uint32_t gen_col;
    uint32_t src_line;
    uint32_t src_col;
};

class SourceMapEmitter {
public:
    void add_mapping(SourceLocation src_loc, uint32_t gen_line, uint32_t gen_col);
    std::string generate_v3_json(std::string_view src_filename, std::string_view gen_filename) const;
    void emit_line_directive(std::ostringstream& out, SourceLocation loc) const {
        out << "\n#line " << loc.line << " \"" << loc.filename << "\"\n";
    }
private:
    std::vector<MappingCoordinate> m_mappings;
};
```

**Verification Tasks:**

1.  **Coordinate Accuracy:** Lower a statement declared at
    `main.fab:5:10`. Verify that the recorded mapping entry links the
    emitted C++ position to source line 5, column 10.
2.  **Debugger Stepping:** Compile generated C++ code containing `#line`
    directives using `clang++ -g`. Run under GDB/LLDB, trigger an
    intentional assertion failure, and verify that the debugger halts
    and reports the crash at `main.fab`.
3.  **JSON Schema Conformance:** Output the source map as a `.map` JSON
    file. Parse the document using a JSON validator and assert
    conformance to the v3 specification keys (`version`, `sources`,
    `mappings`).

#### Exercise 21.9: Automated Native Compilation and Linking Driver

**Context and Theory:**

A complete compiler does not merely write source code to a disk file; it
acts as a **Toolchain Driver** (see Figure 21–5). It coordinates:

1.  Writing the generated C++ code to a temporary file.
2.  Locating the host system's C++ compiler (`clang++` or `g++`).
3.  Invoking the compiler via system subprocess execution (`fork`/`exec`
    or `std::system`), passing flags:
    - `-std=c++23`: Language standard.
    - `-O3`: Maximum optimization.
    - `-march=native`: Native CPU vectorization.
4.  Linking the resulting object files into a standalone native machine
    executable.

<figure id="fig:toolchain-driver">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI2MDZwdCIgaGVpZ2h0PSI0NHB0Igogdmlld0JveD0iMC4wMCAwLjAw
IDYwNS43OCA0NC4wMCIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIiB4bWxuczp4
bGluaz0iaHR0cDovL3d3dy53My5vcmcvMTk5OS94bGluayI+CjxnIGlkPSJncmFwaDAiIGNsYXNz
PSJncmFwaCIgdHJhbnNmb3JtPSJzY2FsZSgxIDEpIHJvdGF0ZSgwKSB0cmFuc2xhdGUoNCA0MCki
Pgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gYXN0IC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNzPSJub2Rl
Ij4KPHRpdGxlPmFzdDwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2si
IGN4PSIzNC43MyIgY3k9Ii0xOCIgcng9IjM0Ljk2IiByeT0iMTgiLz4KPHRleHQgdGV4dC1hbmNo
b3I9Im1pZGRsZSIgeD0iMzQuNzMiIHk9Ii0xNS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcs
IERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+WyBBU1QgXTwv
dGV4dD4KPC9nPgo8IS0tIENwcEVtaXR0ZXIgLS0+CjxnIGlkPSJub2RlMiIgY2xhc3M9Im5vZGUi
Pgo8dGl0bGU+Q3BwRW1pdHRlcjwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGN4PSIxMzYuNzkiIGN5PSItMTgiIHJ4PSI0NS4xNSIgcnk9IjE4Ii8+Cjx0ZXh0IHRl
eHQtYW5jaG9yPSJtaWRkbGUiIHg9IjEzNi43OSIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJDb3Vy
aWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5D
cHBFbWl0dGVyPC90ZXh0Pgo8L2c+CjwhLS0gYXN0JiM0NTsmZ3Q7Q3BwRW1pdHRlciAtLT4KPGcg
aWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5hc3QmIzQ1OyZndDtDcHBFbWl0dGVyPC90
aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTY5LjYxLC0xOEM3My4y
OCwtMTggNzcuMDksLTE4IDgwLjkzLC0xOCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tl
PSJibGFjayIgcG9pbnRzPSI4MS4xMywtMjEuNSA5MS4xMywtMTggODEuMTMsLTE0LjUgODEuMTMs
LTIxLjUiLz4KPC9nPgo8IS0tIHRlbXAgLS0+CjxnIGlkPSJub2RlMyIgY2xhc3M9Im5vZGUiPgo8
dGl0bGU+dGVtcDwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4
PSIyNDIuMzgiIGN5PSItMTgiIHJ4PSIzOC4wMyIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9y
PSJtaWRkbGUiIHg9IjI0Mi4zOCIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywg
RGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj50ZW1wLmNwcDwv
dGV4dD4KPC9nPgo8IS0tIENwcEVtaXR0ZXImIzQ1OyZndDt0ZW1wIC0tPgo8ZyBpZD0iZWRnZTIi
IGNsYXNzPSJlZGdlIj4KPHRpdGxlPkNwcEVtaXR0ZXImIzQ1OyZndDt0ZW1wPC90aXRsZT4KPHBh
dGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTE4Mi4zMiwtMThDMTg2LjE3LC0xOCAx
OTAuMDcsLTE4IDE5My45MywtMTgiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxh
Y2siIHBvaW50cz0iMTk0LjA0LC0yMS41IDIwNC4wNCwtMTggMTk0LjA0LC0xNC41IDE5NC4wNCwt
MjEuNSIvPgo8L2c+CjwhLS0gY2xhbmcgLS0+CjxnIGlkPSJub2RlNCIgY2xhc3M9Im5vZGUiPgo8
dGl0bGU+Y2xhbmc8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBj
eD0iMzkzLjg5IiBjeT0iLTE4IiByeD0iOTEuNDkiIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0LWFuY2hv
cj0ibWlkZGxlIiB4PSIzOTMuODkiIHk9Ii0xNS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcs
IERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+Y2xhbmcrKyAm
IzQ1O3N0ZD1jKysyMyAmIzQ1O08zIDwvdGV4dD4KPC9nPgo8IS0tIHRlbXAmIzQ1OyZndDtjbGFu
ZyAtLT4KPGcgaWQ9ImVkZ2UzIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT50ZW1wJiM0NTsmZ3Q7Y2xh
bmc8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMjgwLjcyLC0x
OEMyODQuNDYsLTE4IDI4OC4zNiwtMTggMjkyLjM4LC0xOCIvPgo8cG9seWdvbiBmaWxsPSJibGFj
ayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIyOTIuNTcsLTIxLjUgMzAyLjU3LC0xOCAyOTIuNTcs
LTE0LjUgMjkyLjU3LC0yMS41Ii8+CjwvZz4KPCEtLSBiaW5hcnkgLS0+CjxnIGlkPSJub2RlNSIg
Y2xhc3M9Im5vZGUiPgo8dGl0bGU+YmluYXJ5PC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIg
c3Ryb2tlPSJibGFjayIgY3g9IjU1Mi40NiIgY3k9Ii0xOCIgcng9IjQ1LjE1IiByeT0iMTgiLz4K
PHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iNTUyLjQ2IiB5PSItMTUuNSIgZm9udC1mYW1p
bHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0i
MTAuMDAiPmJpbmFyeS5vdXQ8L3RleHQ+CjwvZz4KPCEtLSBjbGFuZyYjNDU7Jmd0O2JpbmFyeSAt
LT4KPGcgaWQ9ImVkZ2U0IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5jbGFuZyYjNDU7Jmd0O2JpbmFy
eTwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik00ODUuNTEsLTE4
QzQ4OS4zNCwtMTggNDkzLjE0LC0xOCA0OTYuODYsLTE4Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNr
IiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjQ5Ny4wOSwtMjEuNSA1MDcuMDksLTE4IDQ5Ny4wOSwt
MTQuNSA0OTcuMDksLTIxLjUiLz4KPC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 21–5: Complete Faberge Toolchain Driver</figcaption>
</figure>

**Specification:**

Create a `NativeCompilerDriver` class. Implement subprocess invocation
to compile emitted C++ source strings into executable binaries directly
from the Faberge executable.

``` cpp
class NativeCompilerDriver {
public:
    struct CompileOptions {
        std::string output_binary{"a.out"};
        std::string optimization_level{"-O3"};
        bool generate_debug_symbols{false};
    };

    static bool compile_to_binary(std::string_view cpp_source, const CompileOptions& options) {
        std::string temp_cpp = "/tmp/faberge_temp.cpp";
        std::ofstream(temp_cpp) << cpp_source;

        std::string cmd = "clang++ -std=c++23 " + options.optimization_level + " ";
        if (options.generate_debug_symbols) cmd += "-g ";
        cmd += temp_cpp + " -o " + options.output_binary;

        int status = std::system(cmd.c_str());
        std::remove(temp_cpp.c_str());
        return (status == 0);
    }
};
```

**Verification Tasks:**

1.  **Subprocess Compilation:** Feed a generated C++ string into
    `NativeCompilerDriver::compile_to_binary()`. Verify that the driver
    returns `true` and that the binary exists on disk.
2.  **Binary Execution:** Execute the produced binary using
    `std::system("./a.out")`. Verify that the program runs and returns
    exit status 0.
3.  **Compilation Failure Handling:** Feed malformed C++ code into the
    driver. Verify that `compile_to_binary()` returns `false` without
    throwing unhandled exceptions.

#### Exercise 21.10: End-to-End Compiler Parity Fuzzing

**Context and Theory:**

How do compiler engineers prove that target code generation preserves
the mathematical semantics of the original language?

The gold standard is **Differential Execution Parity Testing** (see
Figure 21–6):

1.  Construct an in-memory **AST Reference Interpreter** that evaluates
    Faberge AST nodes directly using recursive tree-walk evaluation.
2.  Compile the exact same AST through the full `CppEmitter` $\to$
    `clang++` pipeline into a machine binary.
3.  Run both implementations on identical inputs and assert that the
    outputs match bit-for-bit:

$$\text{Eval}_{\text{Interpreter}}(\text{AST}) \equiv \text{Run}_{\text{Native}}(\text{CompiledBinary})$$

If the native binary produces an output that differs from the reference
interpreter, a code generation bug exists.

<figure id="fig:interpreter-evaluation">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzNjhwdCIgaGVpZ2h0PSIxOTBwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzNjguMDAgMTkwLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDE4
NikiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gc291cmNlIC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNz
PSJub2RlIj4KPHRpdGxlPnNvdXJjZTwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIHBvaW50cz0iMzYwLC0xODIgMCwtMTgyIDAsLTE2MCAzNjAsLTE2MCAzNjAsLTE4
MiIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxODAiIHk9Ii0xNjguNSIgZm9udC1m
YW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6
ZT0iMTAuMDAiPkZhYmVyZ2UgU291cmNlIFNjcmlwdDwvdGV4dD4KPC9nPgo8IS0tIGludGVycHJl
dGVyIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmludGVycHJldGVyPC90
aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxNzEsLTEz
OCAyNywtMTM4IDI3LC0xMTYgMTcxLC0xMTYgMTcxLC0xMzgiLz4KPHRleHQgdGV4dC1hbmNob3I9
Im1pZGRsZSIgeD0iOTkiIHk9Ii0xMjQuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWph
VnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPkFTVCBJbnRlcnByZXRl
cjwvdGV4dD4KPC9nPgo8IS0tIHNvdXJjZSYjNDU7Jmd0O2ludGVycHJldGVyIC0tPgo8ZyBpZD0i
ZWRnZTEiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPnNvdXJjZSYjNDU7Jmd0O2ludGVycHJldGVyPC90
aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTE2MC4zOSwtMTU5Ljgz
QzE1MC41NCwtMTU0LjczIDEzOC40MiwtMTQ4LjQ0IDEyNy41NiwtMTQyLjgxIi8+Cjxwb2x5Z29u
IGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjEyOC45MiwtMTM5LjU3IDExOC40
MywtMTM4LjA3IDEyNS42OSwtMTQ1Ljc4IDEyOC45MiwtMTM5LjU3Ii8+CjwvZz4KPCEtLSBjcHBf
ZW1pdHRlciAtLT4KPGcgaWQ9Im5vZGUzIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5jcHBfZW1pdHRl
cjwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMzMz
LC0xMzggMTg5LC0xMzggMTg5LC0xMTYgMzMzLC0xMTYgMzMzLC0xMzgiLz4KPHRleHQgdGV4dC1h
bmNob3I9Im1pZGRsZSIgeD0iMjYxIiB5PSItMTI0LjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5l
dywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5DcHBFbWl0
dGVyIFBhc3M8L3RleHQ+CjwvZz4KPCEtLSBzb3VyY2UmIzQ1OyZndDtjcHBfZW1pdHRlciAtLT4K
PGcgaWQ9ImVkZ2UyIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5zb3VyY2UmIzQ1OyZndDtjcHBfZW1p
dHRlcjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xOTkuNjEs
LTE1OS44M0MyMDkuNDYsLTE1NC43MyAyMjEuNTgsLTE0OC40NCAyMzIuNDQsLTE0Mi44MSIvPgo8
cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIyMzQuMzEsLTE0NS43
OCAyNDEuNTcsLTEzOC4wNyAyMzEuMDgsLTEzOS41NyAyMzQuMzEsLTE0NS43OCIvPgo8L2c+Cjwh
LS0gb3V0cHV0MSAtLT4KPGcgaWQ9Im5vZGU0IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5vdXRwdXQx
PC90aXRsZT4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iOTkiIHk9Ii00MC41IiBmb250
LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1z
aXplPSIxMC4wMCI+T3V0cHV0OiAxNDUzPC90ZXh0Pgo8L2c+CjwhLS0gaW50ZXJwcmV0ZXImIzQ1
OyZndDtvdXRwdXQxIC0tPgo8ZyBpZD0iZWRnZTMiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmludGVy
cHJldGVyJiM0NTsmZ3Q7b3V0cHV0MTwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGQ9Ik05OSwtMTE1Ljk1Qzk5LC0xMDIuMTggOTksLTc3LjI5IDk5LC02MC43Ii8+Cjxw
b2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjEwMi41LC02MC4zNyA5
OSwtNTAuMzcgOTUuNSwtNjAuMzcgMTAyLjUsLTYwLjM3Ii8+CjwvZz4KPCEtLSBjbGFuZyAtLT4K
PGcgaWQ9Im5vZGU1IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5jbGFuZzwvdGl0bGU+Cjxwb2x5Z29u
IGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMzMzLC05NCAxODksLTk0IDE4OSwt
NzIgMzMzLC03MiAzMzMsLTk0Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjI2MSIg
eT0iLTgwLjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9u
b3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5jbGFuZysrICYjNDU7Jmd0OyBOYXRpdmU8L3RleHQ+
CjwvZz4KPCEtLSBjcHBfZW1pdHRlciYjNDU7Jmd0O2NsYW5nIC0tPgo8ZyBpZD0iZWRnZTQiIGNs
YXNzPSJlZGdlIj4KPHRpdGxlPmNwcF9lbWl0dGVyJiM0NTsmZ3Q7Y2xhbmc8L3RpdGxlPgo8cGF0
aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMjYxLC0xMTUuODNDMjYxLC0xMTIuMzIg
MjYxLC0xMDguMjYgMjYxLC0xMDQuMjQiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0i
YmxhY2siIHBvaW50cz0iMjY0LjUsLTEwNC4wNyAyNjEsLTk0LjA3IDI1Ny41LC0xMDQuMDcgMjY0
LjUsLTEwNC4wNyIvPgo8L2c+CjwhLS0gZXF1YWwgLS0+CjxnIGlkPSJub2RlNyIgY2xhc3M9Im5v
ZGUiPgo8dGl0bGU+ZXF1YWw8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJs
YWNrIiBwb2ludHM9IjM2MCwtMTQgMCwtMTQgMCwwIDM2MCwwIDM2MCwtMTQiLz4KPHRleHQgdGV4
dC1hbmNob3I9Im1pZGRsZSIgeD0iMTgwIiB5PSItNC41IiBmb250LWZhbWlseT0iQ291cmllciBO
ZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+QXNzZXJ0
IEVxdWFsPC90ZXh0Pgo8L2c+CjwhLS0gb3V0cHV0MSYjNDU7Jmd0O2VxdWFsIC0tPgo8ZyBpZD0i
ZWRnZTYiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPm91dHB1dDEmIzQ1OyZndDtlcXVhbDwvdGl0bGU+
CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xMTMuNDgsLTM1LjkyQzEyNS4z
NSwtMzAuOTQgMTQyLjM3LC0yMy43OSAxNTYuMjMsLTE3Ljk4Ii8+Cjxwb2x5Z29uIGZpbGw9ImJs
YWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE1Ny43OCwtMjEuMTIgMTY1LjY0LC0xNC4wMyAx
NTUuMDcsLTE0LjY3IDE1Ny43OCwtMjEuMTIiLz4KPC9nPgo8IS0tIG91dHB1dDIgLS0+CjxnIGlk
PSJub2RlNiIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+b3V0cHV0MjwvdGl0bGU+Cjx0ZXh0IHRleHQt
YW5jaG9yPSJtaWRkbGUiIHg9IjI2MSIgeT0iLTQwLjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5l
dywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5PdXRwdXQ6
IDE0NTM8L3RleHQ+CjwvZz4KPCEtLSBjbGFuZyYjNDU7Jmd0O291dHB1dDIgLS0+CjxnIGlkPSJl
ZGdlNSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+Y2xhbmcmIzQ1OyZndDtvdXRwdXQyPC90aXRsZT4K
PHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTI2MSwtNzEuNjJDMjYxLC02OC4x
OSAyNjEsLTY0LjI5IDI2MSwtNjAuNTIiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0i
YmxhY2siIHBvaW50cz0iMjY0LjUsLTYwLjIxIDI2MSwtNTAuMjEgMjU3LjUsLTYwLjIxIDI2NC41
LC02MC4yMSIvPgo8L2c+CjwhLS0gb3V0cHV0MiYjNDU7Jmd0O2VxdWFsIC0tPgo8ZyBpZD0iZWRn
ZTciIGNsYXNzPSJlZGdlIj4KPHRpdGxlPm91dHB1dDImIzQ1OyZndDtlcXVhbDwvdGl0bGU+Cjxw
YXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0yNDYuNTIsLTM1LjkyQzIzNC42NSwt
MzAuOTQgMjE3LjYzLC0yMy43OSAyMDMuNzcsLTE3Ljk4Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNr
IiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjIwNC45MywtMTQuNjcgMTk0LjM2LC0xNC4wMyAyMDIu
MjIsLTIxLjEyIDIwNC45MywtMTQuNjciLz4KPC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 21–6: AST Reference Interpreter</figcaption>
</figure>

**Specification:**

Implement a `CompilerParityTestHarness`. Generate randomized functional
Faberge scripts (incorporating arithmetic, higher-order functions, and
conditionals), evaluate them with a tree-walk interpreter, compile them
to native C++ executables, and assert output equivalence.

``` cpp
class CompilerParityTestHarness {
public:
    static bool verify_program_parity(std::string_view source_code) {
        Lexer lex(source_code);
        auto tokens = lex.tokenise();
        Parser parser(std::move(tokens), source_code);
        auto ast = parser.parse_program();

        // 1. Evaluate via Reference Interpreter
        int64_t expected_output = ReferenceTreeInterpreter::evaluate(*ast);

        // 2. Compile to Native C++
        CppEmitter emitter;
        std::string cpp_code = emitter.generate(*ast);
        NativeCompilerDriver::compile_to_binary(cpp_code, {"/tmp/parity_test.bin"});

        // 3. Execute Native Binary and Capture Exit Code
        int native_exit_code = std::system("/tmp/parity_test.bin");
        int actual_output = WEXITSTATUS(native_exit_code);

        return expected_output == actual_output;
    }
};
```

**Verification Tasks:**

1.  **Mathematical Pipeline Parity:** Test a script calculating
    recursive Fibonacci and factorials. Verify that the reference
    interpreter and the compiled native binary yield the identical
    integer result.
2.  **Lexical Shadowing Parity:** Test a script with nested block
    expressions and variable shadowing. Verify that both execution
    strategies resolve variable bindings identically.
3.  **Automated Campaign:** Run a campaign of $100$ randomized
    functional scripts through `verify_program_parity()`. Verify $100\%$
    semantic parity between interpreted and native execution.